# Data Preprocessing

This notebook turns the raw KKBox files into the tabular dataset the w1 models train on.
One output row is one target `msno` at a cutoff: the features only see history before the
cutoff, and the label says whether that user churned the following month.

The transformation code lives in `src/data_processing/`; the notebook just drives it.

## The data

Four source tables, all keyed by `msno` (a hashed user id):

| Table | One row per | Columns we use |
|---|---|---|
| `members_v3` | user | `city`, `bd` (age), `registered_via`, `registration_init_time`, `gender` |
| `transactions` / `transactions_v2` | payment | plan days, list and paid price, renew/cancel flags, transaction and expiry dates |
| `user_logs` / `user_logs_v2` | user-day | seven listening counters (`num_25` … `total_secs`) |
| `train` / `train_v2` | labelled user | `is_churn` |

The transaction and listening releases each come in two files; together they cover the three cutoffs.

## Pipeline

```mermaid
flowchart LR
    raw["data/*.csv.7z"] --> pq["data/parquet/*.parquet"]
    pq --> users["target users<br/>msno → uid"]
    users --> mem["members"]
    users --> txn["transactions"]
    users --> logs["user logs"]
    mem --> mf["member features"]
    txn --> tf["transaction features"]
    logs --> lf["listening features"]
    mf --> join["left join on uid"]
    tf --> join
    lf --> join
    join --> lab["attach is_churn"]
    lab --> out["train_feb / train / test .parquet"]
```

## Cohorts

| Cohort | Feature cutoff | Label month | Label source | Role |
|---|---|---|---|---|
| `train_feb` | 2017-01-31 | February 2017 | `train.csv` | fit / tune |
| `train` | 2017-02-28 | March 2017 | `train_v2.csv` | cross-month holdout |
| `test` | 2017-03-31 | April 2017 | `sample_submission_v2.csv` | final scoring |

Having two labelled cohorts lets us validate month over month: learn on February, judge on
March, the same way we would predict April.

## Glossary

- `msno` — hashed user id from the source files.
- `uid` — integer code we assign over the union of all users; shared by the three cohorts.
- `row` — position of the user in its own source file. Output rows follow this order.
- `cutoff` — last date a feature is allowed to look at.
- `is_churn` — target: 1 if the user churned in the label month.

## Output

`train_feb.parquet`, `train.parquet` and `test.parquet` (one row per user) plus
`preprocess_params.json` and `metadata.json`, all under `export-tabular-v5/`.
Model training and metrics are in `w1_tabular_models.ipynb`.

**Contents:** [configuration](#1-configuration) · [sources](#2-raw-sources-and-parquet) ·
[users](#3-target-users-and-calendar) · [members](#4-members) ·
[transactions](#5-transactions-and-deduplication) · [aggregates](#6-transaction-features) ·
[listening](#7-daily-listening) · [parameters](#8-listening-features-and-parameters) ·
[joins](#9-final-user-table) · [export](#10-tabular-export).


## 1. Configuration

`KKBOX_SAMPLE_USERS=100` runs a small, isolated export in `export-tabular-v5-sample/` (the first
N users of each cohort). The normal run uses all users and writes `export-tabular-v5/`.
The three cutoffs are the v5 contract. Reusing the v2 export is not supported here because v2
has no February cohort. Earlier exports (`export-tabular-v3/`) are left untouched, so the w2
notebooks keep working on them.


In [29]:
from pathlib import Path
import hashlib
import json
import shutil

import pandas as pd
import polars as pl


In [30]:
# --- Cohorts and calendar --------------------------------------------------
# train_feb: train.csv (v1) labels February churn from history through January 31.
# train:     train_v2.csv labels March churn from history through February 28.
# test:      sample_submission_v2.csv is scored for April from history through March 31.
CUTOFF = {"train_feb": "2017-01-31", "train": "2017-02-28", "test": "2017-03-31"}
LABEL_MONTHS = {"train_feb": "2017-02", "train": "2017-03", "test": "2017-04"}
LABEL_FILES = {"train_feb": "train", "train": "train_v2"}   # cohort -> labelled parquet
SPLITS = ["train_feb", "train", "test"]
DATASET_VERSION = 5


In [31]:
import os

DATA_DIR = Path('../data')
PARQUET_DIR = DATA_DIR / 'parquet'
SAMPLE_USERS = int(os.environ['KKBOX_SAMPLE_USERS']) if os.environ.get('KKBOX_SAMPLE_USERS') else None
if SAMPLE_USERS is not None and SAMPLE_USERS < 1:
    raise ValueError('KKBOX_SAMPLE_USERS must be positive.')
if os.environ.get('KKBOX_REUSE_V2', '0') == '1':
    raise ValueError('KKBOX_REUSE_V2 is not supported by the v5 export (v2 has no February cohort).')
EXPORT_DIR = Path('export-tabular-v5-sample' if SAMPLE_USERS is not None else 'export-tabular-v5')
if shutil.disk_usage(EXPORT_DIR.resolve().parent).free < 2_000_000_000:
    raise OSError('Insufficient free disk for the tabular export (~2 GB).')
print('Building member, transaction and listening features from parquet.')
print('Export directory:', EXPORT_DIR, '| sample users per cohort:', SAMPLE_USERS)


Building member, transaction and listening features from parquet.
Export directory: export-tabular-v5 | sample users per cohort: None


## 2. Raw sources and parquet

The archives ship compressed as `data/*.csv.7z`. `convert_raw_to_parquet` extracts one at a
time, keeps the columns we care about and writes `data/parquet/*.parquet` (zstd). Existing
files are skipped, so re-running is cheap.

```mermaid
flowchart LR
    a["train.csv.7z"] --> b["train.csv"]
    b --> c["train.parquet"]
```

| Archive | One row per | Fields kept |
|---|---|---|
| `train`, `train_v2`, `sample_submission_v2` | user | `msno`, `is_churn` |
| `members_v3` | user | `city`, `registered_via`, `bd`, `registration_init_time`, `gender` |
| `transactions`, `transactions_v2` | payment | the eight fields below |
| `user_logs`, `user_logs_v2` | user-day | `date` + the seven listening counters |

The conversion contract is in `schemas.py`. Each column gets the smallest type that fits the
observed range (`city` → `Int8`, `bd` → `Int16`, prices → `Int16`, `total_secs` → `Float64`).
This step only selects and casts; it never imputes or clips.

The eight transaction fields, used from here on:

```
payment_method_id  payment_plan_days  plan_list_price  actual_amount_paid
is_auto_renew      transaction_date   membership_expire_date  is_cancel
```


In [32]:
import sys

# Make the `w1` package importable when the notebook runs from `workshops/w1`.
sys.path.insert(0, str(Path.cwd().parent))

from w1.src.data_processing.utils import window_start, convert_raw_to_parquet
from w1.src.data_processing.users import load_target_users, users_with_codes
from w1.src.data_processing.members import scan_members
from w1.src.data_processing.transactions import scan_transactions, build_transaction_features
from w1.src.data_processing.user_logs import build_user_day_logs
from w1.src.data_processing.params import fit_preprocess_params, save_preprocess_params
from w1.src.data_processing.pipeline import build_features, attach_labels
from w1.src.data_processing.constants import CATEGORICAL


In [33]:
convert_raw_to_parquet(DATA_DIR)
print('Parquet files:', [p.name for p in sorted(PARQUET_DIR.glob('*.parquet'))])
print('Train schema:', pl.scan_parquet(PARQUET_DIR / 'train_v2.parquet').collect_schema())

[skip] train.parquet already exists
[skip] train_v2.parquet already exists
[skip] sample_submission_v2.parquet already exists
[skip] members_v3.parquet already exists
[skip] transactions.parquet already exists
[skip] transactions_v2.parquet already exists
[skip] user_logs.parquet already exists
[skip] user_logs_v2.parquet already exists
Parquet files: ['members_v3.parquet', 'sample_submission_v2.parquet', 'train.parquet', 'train_v2.parquet', 'transactions.parquet', 'transactions_v2.parquet', 'user_logs.parquet', 'user_logs_v2.parquet']
Train schema: Schema({'msno': String, 'is_churn': Int8})


## 3. Target users and calendar

| Cohort | Monthly window (3 months) | Known through | Target month |
|---|---|---|---|
| `train_feb` | 2016-11-01 to 2017-01-31 | 2017-01-31 | February 2017 |
| `train` | 2016-12-01 to 2017-02-28 | 2017-02-28 | March 2017 |
| `test` | 2017-01-01 to 2017-03-31 | 2017-03-31 | April 2017 |

Monthly listening uses calendar months. `msno` comes from the source; we add `uid` (shared
across cohorts) and keep `row` (position in each cohort's file). Most users appear in more
than one cohort — same subscriber, different cutoff, different label — which is what happens
in production when we score every month. Output rows follow the source file order.

Building the index from the three label files:

```
train.csv     A, B
train_v2.csv  B, C
sample_sub    C, D
        ↓  union, first-seen order
  uid | msno | train_feb_row | train_row | test_row
   0  |  A   | 0             | –         | –
   1  |  B   | 1             | 0         | –
   2  |  C   | –             | 1         | 0
   3  |  D   | –             | –         | 1
```

**Label source.** February labels are the official `train.csv`. A Polars port of
`data/WSDMChurnLabeller.scala` was checked against both files: it agrees on ≈ 98.8 % (February)
and ≈ 97.0 % (March) of the shared users, but ~137k official users per month cannot be
reproduced from the published transactions. We use the official labels as given. Their
prevalence differs (≈ 6.4 % February, ≈ 9.0 % March), and that shift is part of what the
cross-month holdout measures.


In [34]:
labelled_users, test_users, user_index = load_target_users(PARQUET_DIR, LABEL_FILES)

# This is only used for debugging and testing, to reduce the number of users per cohort. 
# The union index is still built over all users, so the ``uid`` values are consistent with the full dataset
if SAMPLE_USERS is not None:
    labelled_users = {name: frame.head(SAMPLE_USERS) for name, frame in labelled_users.items()}
    test_users = test_users.head(SAMPLE_USERS)
    selected = pl.concat([frame.select('msno') for frame in [*labelled_users.values(), test_users]]).unique()
    user_index = user_index.join(selected, on='msno', how='semi')


source_users = {**labelled_users, 'test': test_users}
users_by_split = {name: users_with_codes(source_users[name], user_index) for name in SPLITS}

# Check if the ``msno`` values are unique within each cohort, and print the number of users and churn rate (if labelled).
for name in SPLITS:
    assert source_users[name]['msno'].is_unique().all(), f'{name} has duplicated msno'
    churn = (f"| churn rate: {labelled_users[name]['is_churn'].mean():.4f}"
             if name in labelled_users else '| unlabelled')
    print(f'{name:9s} users: {source_users[name].height:>9,}', churn)

print('unique uid over all cohorts:', user_index.height)
print('train_feb users also in March cohort:',
      users_by_split['train_feb'].join(users_by_split['train'], on='uid', how='semi').height)

for split, cutoff in CUTOFF.items():
    print(split, 'monthly start:', window_start(cutoff).date(), 'cutoff:', cutoff,
          'label month:', LABEL_MONTHS[split])


train_feb users:   992,931 | churn rate: 0.0639
train     users:   970,960 | churn rate: 0.0899
test      users:   907,471 | unlabelled
unique uid over all cohorts: 1143472
train_feb users also in March cohort: 881701
train_feb monthly start: 2016-11-01 cutoff: 2017-01-31 label month: 2017-02
train monthly start: 2016-12-01 cutoff: 2017-02-28 label month: 2017-03
test monthly start: 2017-01-01 cutoff: 2017-03-31 label month: 2017-04


## 4. Members

We keep members only for the target users, then clean and derive features. The imputation
values (`age_mean`, `registered_via_mode`) are fit once on the February cohort and reused
everywhere, so the March holdout never leaks into preprocessing.

One row before and after:

```
raw member                     clean + features
uid | bd    | registered_via    uid | bd     | bd_missing | registered_via | tenure_days | reg_year | reg_month
 7  | -7168 | -1                7   | 33.4*  | 1          | 9*             | 812         | 2014     | 11
 8  |  34   | 8                 8   | 34.0   | 0          | 8              | 402         | 2015     | 12
```

`*` = filled from the training cohort.

Rules: valid age is 1–100, anything else becomes `age_mean` and sets `bd_missing=1`; a negative
`registered_via` becomes the training mode; `tenure_days = cutoff − registration date`.
`gender` is loaded by the raw schema but not used as a feature.


In [35]:
members = scan_members(PARQUET_DIR, user_index).collect(engine='streaming')
print('Members:', members.shape)
print(members.head(3))


Members: (1018235, 5)
shape: (3, 5)
┌────────┬──────┬────────────────┬─────┬────────────────────────┐
│ uid    ┆ city ┆ registered_via ┆ bd  ┆ registration_init_time │
│ ---    ┆ ---  ┆ ---            ┆ --- ┆ ---                    │
│ u32    ┆ i8   ┆ i8             ┆ i16 ┆ i32                    │
╞════════╪══════╪════════════════╪═════╪════════════════════════╡
│ 507847 ┆ 1    ┆ 7              ┆ 0   ┆ 20160119               │
│ 917388 ┆ 1    ┆ 7              ┆ 0   ┆ 20160121               │
│ 275968 ┆ 1    ┆ 7              ┆ 0   ┆ 20160125               │
└────────┴──────┴────────────────┴─────┴────────────────────────┘


## 5. Transactions and deduplication

Both releases are stacked, cut at the cutoff and reduced to the target users. Exact duplicates
are then dropped on `msno` plus the eight transaction fields — the first copy wins. Two payments
on the same day with different values are different rows and both stay.

Suppose user `A` has these three rows after the cutoff filter:

| msno | transaction_date | payment_plan_days | actual_amount_paid | is_cancel |
|---|---|---|---|---|
| A | 20170120 | 30 | 149 | 0 |
| A | 20170120 | 30 | 149 | 0 |
| A | 20170120 | 30 | 180 | 0 |

The first two are identical in all eight fields, so one is dropped. The third differs in
`actual_amount_paid`, so it survives. Two rows remain:

| msno | transaction_date | payment_plan_days | actual_amount_paid | is_cancel |
|---|---|---|---|---|
| A | 20170120 | 30 | 149 | 0 |
| A | 20170120 | 30 | 180 | 0 |

`_ord` keeps the stable source order and is used later as a tie-break — it is not a time.
Records after the cutoff never influence earlier state.


In [36]:
transactions = scan_transactions(
    PARQUET_DIR, user_index, CUTOFF['test']
).collect(engine='streaming')
print('Transactions:', transactions.shape)
print(transactions.select('uid', '_ord', 'transaction_date', 'actual_amount_paid', 'is_cancel').head(3))


Transactions: (17858127, 10)
shape: (3, 5)
┌────────┬──────┬──────────────────┬────────────────────┬───────────┐
│ uid    ┆ _ord ┆ transaction_date ┆ actual_amount_paid ┆ is_cancel │
│ ---    ┆ ---  ┆ ---              ┆ ---                ┆ ---       │
│ u32    ┆ u32  ┆ i32              ┆ i16                ┆ i8        │
╞════════╪══════╪══════════════════╪════════════════════╪═══════════╡
│ 110581 ┆ 0    ┆ 20150930         ┆ 129                ┆ 0         │
│ 724815 ┆ 1    ┆ 20150930         ┆ 149                ┆ 0         │
│ 789025 ┆ 4    ┆ 20150930         ┆ 149                ┆ 0         │
└────────┴──────┴──────────────────┴────────────────────┴───────────┘


## 6. Transaction features

Each user's payment rows are aggregated over **all history up to the cutoff**, not just the
90-day window. Many payment rows per user collapse into a single feature row.

Input — a few of user `A`'s transactions:

| transaction_date | payment_plan_days | actual_amount_paid | is_auto_renew | is_cancel |
|---|---|---|---|---|
| 20161115 | 30 | 149 | 1 | 0 |
| 20161220 | 30 | 149 | 1 | 0 |
| 20170119 | 30 | 149 | 0 | 1 |

Output — **one row for `A`**, with ~28 columns. Grouped:

| group | example columns |
|---|---|
| counts | `transaction_count`, `n_payment_methods`, `n_plans` |
| sums | `sum_amount_paid`, `sum_plan_days`, `sum_list_price` |
| averages | `average_amount_paid`, `average_plan`, `plan_coverage_ratio` |
| renewal / cancel | `is_cancel_change_flag`, `is_auto_renew_change_flag`, `autorenew_not_cancel` |
| last record | `last_payment_method_id`, `last_plan_days`, `last_is_cancel` |
| recency | `days_to_expire`, `days_since_last_transaction`, `expire_dayofweek` |

For the example above: `transaction_count=3`, `sum_amount_paid=447`, `average_plan=30.0`,
`last_is_cancel=1` and `days_since_last_transaction=12` (cutoff 2017-01-31 minus 2017-01-19).

`duration = expiry − transaction_date`, `plan_coverage_ratio = duration / plan_days` when plan
days are positive, and `days_to_expire = last expiry − cutoff` (can be negative). The last
record has the largest `transaction_date`, with `_ord` as tie-break. A user with no payments
keeps nulls until the final join fills them with 0.


In [37]:
transaction_features = build_transaction_features(
    transactions, users_by_split['train_feb'], CUTOFF['train_feb'])
print('Transaction features:', transaction_features.shape)
print(transaction_features.head(3))


Transaction features: (990836, 29)
shape: (3, 29)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ uid    ┆ is_cancel_ ┆ is_auto_re ┆ transacti ┆ … ┆ days_to_e ┆ expire_da ┆ days_sinc ┆ differenc │
│ ---    ┆ change_fla ┆ new_change ┆ on_count  ┆   ┆ xpire     ┆ yofweek   ┆ e_last_tr ┆ e         │
│ u32    ┆ g          ┆ _flag      ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ansaction ┆ ---       │
│        ┆ ---        ┆ ---        ┆ u32       ┆   ┆ i32       ┆ i8        ┆ ---       ┆ f64       │
│        ┆ i8         ┆ i8         ┆           ┆   ┆           ┆           ┆ i32       ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 976026 ┆ 1          ┆ 1          ┆ 24        ┆ … ┆ 19        ┆ 6         ┆ 11        ┆ 6.208333  │
│ 209479 ┆ 0          ┆ 0          ┆ 13        ┆ … ┆ 12        ┆ 6         ┆ 29        ┆ 0.0       │
│ 657379 ┆ 0          ┆ 1          ┆ 13  

## 7. Daily listening

The two log releases are filtered to the needed dates and target users, then grouped to one
row per `(uid, date)`. Negative counts are clipped to 0 and non-finite `total_secs` becomes 0.

```
raw records (msno, date)                       user-day
A | 20170201 | num_25=3 | total_secs=...       uid | date     | num_25 | ... | total_secs | rows
A | 20170201 | num_25=2 | total_secs=...   →   A   | 20170201 | 5      | ... | ...        | 2
A | 20170203 | num_25=1 | total_secs=...       A   | 20170203 | 1      | ... | ...        | 1
```

`rows` is the number of raw records behind the day. A day with a record counts as active even
if every counter is 0. This single `(uid, date)` table feeds both the monthly features here
and the daily sequences in w2.


In [38]:
# One user-day table covers the earliest monthly window (February cohort) to the last cutoff.
LOG_START = min(window_start(cutoff) for cutoff in CUTOFF.values())
user_day = build_user_day_logs(PARQUET_DIR, user_index, LOG_START, CUTOFF['test'])
print('User-day rows:', user_day.height, '| from', LOG_START.date(), 'to', CUTOFF['test'])
print(user_day.head(3))


User-day rows: 72614379 | from 2016-11-01 to 2017-03-31
shape: (3, 10)
┌────────┬──────────┬────────┬────────┬───┬─────────┬─────────┬──────────────┬──────┐
│ uid    ┆ date     ┆ num_25 ┆ num_50 ┆ … ┆ num_100 ┆ num_unq ┆ total_secs   ┆ rows │
│ ---    ┆ ---      ┆ ---    ┆ ---    ┆   ┆ ---     ┆ ---     ┆ ---          ┆ ---  │
│ u32    ┆ i32      ┆ i32    ┆ i32    ┆   ┆ i32     ┆ i32     ┆ f32          ┆ u32  │
╞════════╪══════════╪════════╪════════╪═══╪═════════╪═════════╪══════════════╪══════╡
│ 310019 ┆ 20170102 ┆ 4      ┆ 3      ┆ … ┆ 23      ┆ 28      ┆ 6322.979004  ┆ 1    │
│ 835656 ┆ 20161122 ┆ 0      ┆ 0      ┆ … ┆ 67      ┆ 12      ┆ 19513.849609 ┆ 1    │
│ 817352 ┆ 20161204 ┆ 15     ┆ 4      ┆ … ┆ 33      ┆ 42      ┆ 8912.542969  ┆ 1    │
└────────┴──────────┴────────┴────────┴───┴─────────┴─────────┴──────────────┴──────┘


## 8. Listening features and parameters

Clipping limits and the sequence mean/std come from the February cohort and its window only.
Daily values are grouped to months, the monthly mean is clipped at the daily p99.9 (then the
sum is rebuilt from it), and each user is described by its last three active months.

```mermaid
flowchart LR
    ud["user-day (uid, date)"] --> mo["monthly sums (uid, YYYYMM)"]
    mo --> clip["clip mean at p99.9, rebuild sum"]
    clip --> rank["rank last 3 active months"]
    rank --> wide["wide row: last / last_second / last_third"]
```

`last`, `last_second` and `last_third` are the most recent **active** months, not necessarily
consecutive. `days_since_last_log` is relative to the cutoff; a user with no logs gets
`has_listening_logs=0` and a sentinel recency. The sequence tensors in w2 stay as unscaled
`log1p` values.


In [39]:
# Fit on the training cohort only (February); March is the holdout and April the test.
params = fit_preprocess_params(members, user_day, users_by_split['train_feb'], CUTOFF['train_feb'])
print('Age mean:', round(params['age_mean'], 2),
      '| registered-via mode:', params['registered_via_mode'])
print('Daily clipping limits:', params['log_clip_p999'])


Age mean: 29.97 | registered-via mode: 7
Daily clipping limits: {'num_25': 146.0, 'num_50': 44.0, 'num_75': 18.0, 'num_985': 23.0, 'num_100': 334.0, 'num_unq': 238.0, 'total_secs': 78007.4296875}


## 9. Final user table

`pipeline.build_features` starts from one row per target user and left-joins the three feature
groups on `uid`. Left joins keep users who have no member or listening records, and each group
has at most one row per `uid`, so the result stays one row per user. Nulls become 0, dtypes
shrink, and rows go back to source order via `row`. Only labelled cohorts get `is_churn`.

```mermaid
flowchart TD
    users["users (uid, msno, row)"] --> f["combined features"]
    m["member features (uid)"] -->|left join| f
    t["transaction features (uid)"] -->|left join| f
    l["listening features (uid)"] -->|left join| f
    f --> p["impute by source, shrink dtypes, sort by row"]
```

A final row mixes the three sources:

| feature | comes from | how |
|---|---|---|
| `bd`, `bd_missing`, `has_member_record` | members | valid age or train mean; invalid or absent record flags |
| `transaction_count`, `days_to_expire`, `has_transactions` | transactions | count; last expiry − cutoff; record presence |
| `last_sum_total_songs` | logs | clipped sum in the latest active month |
| `days_since_last_log`, `has_listening_logs` | logs | recency and presence |

The full feature list is in `metadata.json`.


In [40]:
feature_tables = {
    split: build_features(members, transactions, user_day, users_by_split[split],
                          CUTOFF[split], params)
    for split in SPLITS
}
final_tables = attach_labels(feature_tables, labelled_users, test_users)
print({split: frame.shape for split, frame in final_tables.items()})
final_tables['train_feb'].head(3)


{'train_feb': (992931, 108), 'train': (970960, 108), 'test': (907471, 107)}


msno,is_churn,city,registered_via,bd,bd_missing,tenure_days,registration_year,registration_month,is_cancel_change_flag,is_auto_renew_change_flag,transaction_count,average_amount_paid,average_amount_charged,average_plan,n_payment_methods,n_plans,n_list_prices,n_amounts_paid,duration,plan_coverage_ratio,sum_plan_days,sum_list_price,sum_amount_paid,more_than_30,autorenew_not_cancel,not_autorenew_cancel,max_plan_days,last_payment_method_id,last_plan_days,last_is_auto_renew,last_is_cancel,last_discount,days_to_expire,expire_dayofweek,days_since_last_transaction,difference,…,last_second_ratio_100,last_second_ratio_25,sum_last_third_month_25,mean_last_third_month_25,sum_last_third_month_50,mean_last_third_month_50,sum_last_third_month_75,mean_last_third_month_75,sum_last_third_month_985,mean_last_third_month_985,sum_last_third_month_100,mean_last_third_month_100,sum_last_third_month_unq,mean_last_third_month_unq,sum_last_third_month_total_secs,mean_last_third_month_total_secs,last_third_login,last_third_sum_total_songs,last_third_ratio_100,last_third_ratio_25,sum_diff_last_2_month_25,sum_diff_last_2_month_50,sum_diff_last_2_month_75,sum_diff_last_2_month_985,sum_diff_last_2_month_100,sum_diff_last_2_month_unq,sum_diff_last_2_month_total_secs,mean_diff_last_2_month_25,mean_diff_last_2_month_50,mean_diff_last_2_month_75,mean_diff_last_2_month_985,mean_diff_last_2_month_100,mean_diff_last_2_month_unq,mean_diff_last_2_month_total_secs,login_difference,days_since_last_log,has_listening_logs
str,i8,i8,i16,f32,i8,i32,i16,i8,i8,i8,u32,f32,f32,f32,u32,u32,u32,u32,f32,f32,i32,i32,i32,i32,i8,i8,i16,i8,i16,i8,i8,i16,i32,i8,i32,f32,…,f32,f32,i32,f32,i32,f32,i32,f32,i32,f32,i32,f32,i32,f32,i32,f32,i32,i32,f32,f32,i32,i32,i32,i32,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,i32,i32,i8
"""waLDQMmcOu2jLDaV1ddDkgCrB/jl6s…",1,18,9,36.0,0,4318,2005,4,0,0,2,74.5,74.5,18.5,1,2,2,2,18.5,1.0,37,149,149,0,0,0,30,38,30,0,0,0,6,0,24,0.0,…,0.8,0.12,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0.0,0.0,32,15,13,7,310,344,83065,0.444444,0.833333,0.277778,-0.055556,8.333333,8.0,2567.12915,16,2,1
"""QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq…",1,10,9,38.0,0,4317,2005,4,1,1,22,150.409088,143.636368,28.772728,2,3,3,2,49.272728,1.630312,633,3160,3309,21,0,0,31,39,30,1,0,0,49,1,0,6.772727,…,0.833333,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0,0.0,0.0,25,9,4,5,555,285,99806,4.166667,1.416667,0.666667,0.75,91.666664,47.0,16381.495117,2,32,1
"""fGwBva6hikQmTJzrbz/2Ezjm5Cth5j…",1,11,9,27.0,0,4125,2005,10,1,1,10,149.199997,149.199997,30.0,2,1,2,2,31.299999,1.043333,300,1492,1492,6,0,0,30,39,30,1,1,0,3,4,19,0.0,…,0.592827,0.270042,17,4.25,14,3.5,6,1.5,11,2.75,31,7.75,72,18.0,10355,2588.67041,4,79,0.392405,0.21519,-106,-31,-11,-5,-191,-314,-49208,-15.535714,-4.535714,-1.607143,-0.892857,-28.892857,-47.05357,-7448.196289,1,0,1


## 10. Tabular export

The last step writes the contract: one parquet per cohort plus `preprocess_params.json` and
`metadata.json`.

```
export-tabular-v5/
  train_feb.parquet       one row per February-cohort user (features + is_churn)
  train.parquet           one row per March-cohort user    (features + is_churn)
  test.parquet            one row per April-cohort user     (features only)
  preprocess_params.json  imputation + clipping values fitted on train_feb
  metadata.json           cutoffs, label months, feature names, row counts, msno hashes
```

`metadata.json["features"]` is the official feature order the models must follow. It is
written after every table exists, so a half-written directory never looks complete.


In [41]:
def _hash_msno(df):
    digest = hashlib.sha1()
    for value in df["msno"].to_list():
        digest.update(str(value).encode())
        digest.update(b"\n")
    return digest.hexdigest()


def export_tabular_dataset(export_dir, tables, metadata):
    """Write one parquet per cohort (``tables``: name -> frame) and their contract."""
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)
    # Invalidate a previous manifest before overwriting any part of this export.
    (export_dir / "metadata.json").unlink(missing_ok=True)

    for name, frame in tables.items():
        frame.write_parquet(export_dir / f"{name}.parquet", compression="zstd")

    metadata = dict(metadata)
    metadata["dataset_version"] = DATASET_VERSION
    metadata["splits"] = list(tables)
    metadata["features"] = [c for c in tables["train"].columns if c not in ("msno", "is_churn")]
    for name, frame in tables.items():
        metadata[f"n_{name}"] = int(frame.height)
        metadata[f"hash_{name}"] = _hash_msno(frame)
        if "is_churn" in frame.columns:
            metadata[f"churn_rate_{name}"] = float(frame["is_churn"].mean())
    metadata["generated_at"] = pd.Timestamp.now().isoformat(timespec="seconds")
    with open(export_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, default=str)
    return metadata


In [42]:
save_preprocess_params(params, EXPORT_DIR / 'preprocess_params.json')
metadata = export_tabular_dataset(EXPORT_DIR, final_tables, {
    'cutoffs': CUTOFF, 'label_months': LABEL_MONTHS, 'label_files': LABEL_FILES,
    'categorical': CATEGORICAL, 'preprocess_params': params,
    'preprocess_params_fit_on': 'train_feb', 'sample_users': SAMPLE_USERS,
})

print('Export:', EXPORT_DIR.resolve())
for split, frame in final_tables.items():
    print(split, frame.shape)
for path in sorted(EXPORT_DIR.iterdir()):
    if path.is_file():
        print(path.name, round(path.stat().st_size / 1e6, 2), 'MB')


Export: /home/alejandropenagos/Escritorio/Alejandro/Machine Learning/customer-churn-prediction/workshops/w1/export-tabular-v5
train_feb (992931, 108)
train (970960, 108)
test (907471, 107)
metadata.json 0.0 MB
preprocess_params.json 0.0 MB
test.parquet 163.71 MB
train.parquet 174.0 MB
train_feb.parquet 178.3 MB
